#  Bronze Layer: NYC Taxi Data Ingestion

## Objective

Load raw NYC Yellow Taxi trip data from a Unity Catalog Volume into a Delta Lake Bronze table.

The Bronze layer preserves the original source data while adding technical metadata for traceability and reproducibility.

**Technologies:** Databricks, Apache Spark, PySpark, Delta Lake, Unity Catalog.

**Source:** NYC TLC Yellow Taxi, January 2025.

**Target table:** `workspace.nyc_taxi.bronze_trips`

## 1. Read Raw Data from Unity Catalog Volume

### Objective

Load the original NYC TLC Parquet file into a Spark DataFrame.

### Operations
- Define the source file path.
- Read the Parquet file using Apache Spark.
- Inspect the dataset schema.
- Display a sample of five records.

The original source file remains unchanged.

In [0]:
from pathlib import Path
import json

# Unity Catalog RAW Volume
RAW_DIR = Path("/Volumes/workspace/nyc_taxi/raw_data")

# Read ingestion manifest
manifest_path = RAW_DIR / (
    "yellow_tripdata_2025-01.parquet.manifest.json"
)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

# Extract batch metadata
FILE_NAME = manifest["file_name"]
BATCH_ID = manifest["batch_id"]

PERIOD = (
    f"{int(manifest['year'])}-"
    f"{int(manifest['month']):02d}"
)

RAW_PATH = str(RAW_DIR / FILE_NAME)

BRONZE_TABLE = "workspace.nyc_taxi.bronze_trips"

# Load source data
df_raw = spark.read.parquet(RAW_PATH)

print("Source:", RAW_PATH)
print("Batch ID:", BATCH_ID)
print("Period:", PERIOD)
print("Columns:", len(df_raw.columns))

df_raw.printSchema()

display(df_raw.limit(5))

## 2. Inspect Source Data

### Objective

Understand the structure and volume of the dataset before ingestion.

### Operations
- Count the total number of records.
- Identify column names and data types.
- Examine missing values in important fields.
- Verify the presence of required trip attributes.

This inspection establishes a baseline for validating the Bronze ingestion.

In [0]:
from pyspark.sql import functions as F

# Total source records
source_count = df_raw.count()

print("Total source records:", source_count)

# List all columns
print("\nSource columns:")
for column in df_raw.columns:
    print("-", column)

# Required columns for future transformations
required_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "total_amount",
    "PULocationID",
    "DOLocationID"
]

missing_columns = [
    col for col in required_columns
    if col not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Count null values in important columns
null_checks = [
    F.sum(F.col(col).isNull().cast("int")).alias(col)
    for col in required_columns
]

print("\nNull value counts:")
display(df_raw.select(*null_checks))

## 3. Add Ingestion Metadata

### Objective

Enrich the raw dataset with technical metadata to ensure data lineage and batch traceability.

### Metadata
- `source_file`: Original Parquet file name.
- `ingested_at`: Timestamp of the ingestion.
- `batch_id`: Unique identifier of the source batch.
- `source_year_month`: Source reporting period.

Business data remains unchanged in the Bronze layer.

In [0]:
# Add technical metadata
df_bronze = (
    df_raw
    .withColumn(
        "source_file",
        F.lit(FILE_NAME)
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "batch_id",
        F.lit(BATCH_ID)
    )
    .withColumn(
        "source_year_month",
        F.lit(PERIOD)
    )
)

print("Bronze DataFrame prepared.")

print("\nTechnical metadata:")
display(
    df_bronze.select(
        "source_file",
        "ingested_at",
        "batch_id",
        "source_year_month"
    ).limit(5)
)

print("\nBronze schema:")
df_bronze.printSchema()

## 4. Write Data to the Bronze Delta Table

### Objective

Persist the enriched dataset as a managed Delta Lake table.

### Operations
- Select an explicit batch write strategy.
- Write the DataFrame using Delta format.
- Register the table in Unity Catalog.
- Prevent duplicate ingestion of the same monthly batch.

**Target:** `workspace.nyc_taxi.bronze_trips`

The chosen write strategy must support safe batch reprocessing.

In [0]:
# Prevent empty batches
if source_count == 0:
    raise ValueError("Source batch is empty.")

# Check whether Bronze table exists
if not spark.catalog.tableExists(BRONZE_TABLE):

    print("Creating Bronze Delta table...")

    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

else:

    print(f"Reprocessing batch: {PERIOD}")

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"source_year_month = '{PERIOD}'"
        )
        .saveAsTable(BRONZE_TABLE)
    )

print("Bronze Delta write: SUCCESS")
print("Target table:", BRONZE_TABLE)

## 5. Validate the Bronze Layer

### Objective

Verify that the Delta table was created successfully and contains the expected data.

### Validation Checks
- Confirm that the table exists.
- Compare source and Bronze record counts.
- Inspect technical metadata columns.
- Verify that the table uses Delta format.
- Inspect Delta table history.

In [0]:
# Read persisted Bronze Delta table
df_saved = spark.table(BRONZE_TABLE)

# Count records for the current source month
bronze_count = (
    df_saved
    .filter(F.col("source_year_month") == PERIOD)
    .count()
)

print("RAW records:", source_count)
print("Bronze records:", bronze_count)

# Validate record counts
assert bronze_count == source_count, (
    "RAW and Bronze record counts do not match!"
)

print("Row count validation: SUCCESS")

# Inspect technical metadata
display(
    df_saved.select(
        "source_file",
        "batch_id",
        "source_year_month"
    ).distinct()
)

# Verify Delta table format
spark.sql(
    f"DESCRIBE DETAIL {BRONZE_TABLE}"
).select(
    "format",
    "numFiles"
).show(truncate=False)

# Inspect Delta transaction history
display(
    spark.sql(
        f"DESCRIBE HISTORY {BRONZE_TABLE}"
    )
)

## 6. Test Batch Reprocessing and Idempotency

### Objective

Ensure that running the same monthly ingestion multiple times does not create duplicate batches.

### Test Scenario
1. Record the initial Bronze row count.
2. Reprocess the January 2025 batch.
3. Compare row counts before and after.
4. Verify the batch metadata.
5. Document the observed behavior.

### Expected Result

Reprocessing the same batch should not silently duplicate its records.

In [0]:
# Count current monthly Bronze records
before_count = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("source_year_month") == PERIOD)
    .count()
)

print("Before reprocessing:", before_count)

# Reprocess the same batch
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option(
        "replaceWhere",
        f"source_year_month = '{PERIOD}'"
    )
    .saveAsTable(BRONZE_TABLE)
)

# Count records after reprocessing
after_count = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("source_year_month") == PERIOD)
    .count()
)

print("After reprocessing:", after_count)

# Verify the expected result
assert before_count == after_count
assert after_count == source_count

print("Idempotency test: SUCCESS")

## 7. Detect Source Schema Changes

### Objective

Simulate a source schema change and detect new columns before writing to Bronze.

This test does not modify the production Bronze table.

In [0]:
# Simulate an unexpected new source column
df_changed = df_raw.withColumn(
    "unexpected_column",
    F.lit("schema_v2")
)

original_columns = set(df_raw.columns)
changed_columns = set(df_changed.columns)

added_columns = changed_columns - original_columns

print("New columns detected:", added_columns)

assert "unexpected_column" in added_columns

print("Schema drift detection: SUCCESS")
print("Bronze table remains unchanged.")